# NB07: Diverse Organism TMFA Comparison

Expand the thermodynamic impact analysis from 2 reference organisms to 10 diverse
species spanning 6 phyla. For each organism: build a genome-scale model, run FBA
unconstrained, then apply TMFA constraints with both Marvin and OPAM2 dG values.
Compare growth rates, flux distributions, and blocked reactions across organisms
to assess how thermodynamic constraints interact with metabolic diversity.

**Organisms:**
- E. coli K-12 MG1655 (Proteobacteria, enteric)
- B. subtilis 168 (Firmicutes, soil)
- P. putida KT2440 (Proteobacteria, soil/industrial)
- S. aureus NCTC 8325 (Firmicutes, pathogen)
- S. enterica Typhimurium LT2 (Proteobacteria, enteric)
- M. tuberculosis H37Rv (Actinobacteria, pathogen)
- C. acetobutylicum ATCC 824 (Firmicutes, anaerobe)
- Synechocystis sp. PCC 6803 (Cyanobacteria, phototroph)
- K. pneumoniae HS11286 (Proteobacteria, pathogen)
- D. radiodurans R1 (Deinococcota, extremophile)

In [1]:
import json
import sys
import subprocess
import warnings
from pathlib import Path

import cobra
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from modelseedpy.core.mstemplate import MSTemplateBuilder
from modelseedpy.core.msgenome import MSGenome, MSFeature
from modelseedpy import MSBuilder, MSMedia, MSATPCorrection, MSGapfill
from modelseedpy.core.msatpcorrection import load_default_medias
from modelseedpy.core.msmodel import get_reaction_constraints_from_direction
from modelseedpy.fbapkg.tmfapkg import TMFAPkg
from modelseedpy.core.fbahelper import FBAHelper

sys.path.insert(0, '/global_share/KBaseUtilities/KBUtilLib/src')
from kbutillib.bvbrc_utils import BVBRCUtils
BVBRCUtils.save = lambda self, name, obj: None

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

warnings.filterwarnings('ignore', category=FutureWarning)
print('Imports OK')

modelseedpy 0.4.3
Imports OK


## 1. Load Templates

In [2]:
TEMPLATE_DIR = Path('/tmp/ModelSEEDTemplates/templates/v7.0')

if not TEMPLATE_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/ModelSEED/ModelSEEDTemplates.git',
         '/tmp/ModelSEEDTemplates'],
        check=True
    )

templates = {}
for tname, fname in [
    ('GramNeg', 'GramNegModelTemplateV7.json'),
    ('GramPos', 'GramPosModelTemplateV7.json'),
    ('Core', 'Core-V6.json'),
]:
    with open(TEMPLATE_DIR / fname) as f:
        data = json.load(f)
    templates[tname] = MSTemplateBuilder.from_dict(data, None).build()
    t = templates[tname]
    print(f'{tname}: {len(t.reactions)} reactions, {len(t.compounds)} compounds')

GramNeg: 8584 reactions, 6573 compounds


GramPos: 8584 reactions, 6573 compounds
Core: 252 reactions, 197 compounds


## 2. Define Organisms

In [3]:
organisms = {
    'ecoli': {
        'name': 'Escherichia coli K-12 MG1655',
        'bvbrc_id': '511145.12',
        'template': 'GramNeg',
        'phylum': 'Proteobacteria',
        'niche': 'enteric',
    },
    'bsubtilis': {
        'name': 'Bacillus subtilis 168',
        'bvbrc_id': '224308.43',
        'template': 'GramPos',
        'phylum': 'Firmicutes',
        'niche': 'soil',
    },
    'pputida': {
        'name': 'Pseudomonas putida KT2440',
        'bvbrc_id': '160488.4',
        'template': 'GramNeg',
        'phylum': 'Proteobacteria',
        'niche': 'soil/industrial',
    },
    'saureus': {
        'name': 'Staphylococcus aureus NCTC 8325',
        'bvbrc_id': '93061.5',
        'template': 'GramPos',
        'phylum': 'Firmicutes',
        'niche': 'pathogen',
    },
    'senterica': {
        'name': 'Salmonella enterica Typhimurium LT2',
        'bvbrc_id': '99287.12',
        'template': 'GramNeg',
        'phylum': 'Proteobacteria',
        'niche': 'enteric',
    },
    'mtb': {
        'name': 'Mycobacterium tuberculosis H37Rv',
        'bvbrc_id': '83332.12',
        'template': 'GramPos',
        'phylum': 'Actinobacteria',
        'niche': 'pathogen',
    },
    'cace': {
        'name': 'Clostridium acetobutylicum ATCC 824',
        'bvbrc_id': '272562.8',
        'template': 'GramPos',
        'phylum': 'Firmicutes',
        'niche': 'anaerobe',
    },
    'synechocystis': {
        'name': 'Synechocystis sp. PCC 6803 substr. GT-I',
        'bvbrc_id': '1080228.3',
        'template': 'GramNeg',
        'phylum': 'Cyanobacteria',
        'niche': 'phototroph',
    },
    'kpneumoniae': {
        'name': 'Klebsiella pneumoniae HS11286',
        'bvbrc_id': '1125630.4',
        'template': 'GramNeg',
        'phylum': 'Proteobacteria',
        'niche': 'pathogen',
    },
    'dradiodurans': {
        'name': 'Deinococcus radiodurans ATCC 13939',
        'bvbrc_id': '1408434.13',
        'template': 'GramNeg',
        'phylum': 'Deinococcota',
        'niche': 'extremophile',
    },
}

print(f'{len(organisms)} organisms defined')
for org_id, info in organisms.items():
    print(f'  {org_id:15s} {info["template"]:8s} {info["phylum"]:16s} {info["name"]}')

10 organisms defined
  ecoli           GramNeg  Proteobacteria   Escherichia coli K-12 MG1655
  bsubtilis       GramPos  Firmicutes       Bacillus subtilis 168
  pputida         GramNeg  Proteobacteria   Pseudomonas putida KT2440
  saureus         GramPos  Firmicutes       Staphylococcus aureus NCTC 8325
  senterica       GramNeg  Proteobacteria   Salmonella enterica Typhimurium LT2
  mtb             GramPos  Actinobacteria   Mycobacterium tuberculosis H37Rv
  cace            GramPos  Firmicutes       Clostridium acetobutylicum ATCC 824
  synechocystis   GramNeg  Cyanobacteria    Synechocystis sp. PCC 6803 substr. GT-I
  kpneumoniae     GramNeg  Proteobacteria   Klebsiella pneumoniae HS11286
  dradiodurans    GramNeg  Deinococcota     Deinococcus radiodurans ATCC 13939


## 3. Fetch Genomes from BV-BRC

In [4]:
bvbrc = BVBRCUtils()

def build_msgenome(genome_dict):
    """Convert BV-BRC genome dict to MSGenome with RAST ontology terms."""
    genome = MSGenome()
    genome.id = genome_dict['id']
    genome.scientific_name = genome_dict['scientific_name']
    genome.domain = genome_dict.get('domain', 'Bacteria')
    features = []
    func_count = 0
    for feat_dict in genome_dict['features']:
        seq = feat_dict.get('protein_translation', '')
        if not seq:
            continue
        funcs = feat_dict.get('functions', [])
        desc = ' / '.join(funcs) if funcs else ''
        aliases = feat_dict.get('aliases', [])
        f = MSFeature(feat_dict['id'], seq, description=desc, aliases=aliases)
        f.ontology_terms = {}
        roles = []
        for func_str in funcs:
            for role in func_str.split(' / '):
                role = role.strip()
                if role and role != 'hypothetical protein':
                    roles.append(role)
        if roles:
            f.ontology_terms['RAST'] = roles
            func_count += 1
        features.append(f)
    genome.add_features(features)
    return genome, func_count

genomes = {}
for org_id, org_info in organisms.items():
    cache_path = DATA_DIR / f'genome_{org_id}.json'
    if cache_path.exists():
        print(f'Loading cached genome for {org_info["name"]}...')
        with open(cache_path) as f:
            genome_dict = json.load(f)
    else:
        print(f'Downloading {org_info["name"]} ({org_info["bvbrc_id"]}) from BV-BRC...')
        try:
            genome_dict = bvbrc.build_kbase_genome_from_api(org_info['bvbrc_id'])
        except Exception as e:
            print(f'  ERROR downloading {org_id}: {e}')
            continue
        with open(cache_path, 'w') as f:
            json.dump(genome_dict, f)
    
    genome, func_count = build_msgenome(genome_dict)
    genomes[org_id] = genome
    print(f'  {org_id}: {len(genome.features)} features, {func_count} with functional annotations')

print(f'\nLoaded {len(genomes)}/{len(organisms)} genomes')

2026-05-13 18:22:49,310 - kbutillib.bvbrc_utils.BVBRCUtils - INFO - Loaded kbase tokens from /home/freiburgermsu/.kbase/token


Loading cached genome for Escherichia coli K-12 MG1655...


  ecoli: 8870 features, 8353 with functional annotations
Loading cached genome for Bacillus subtilis 168...


  bsubtilis: 8491 features, 6811 with functional annotations
Loading cached genome for Pseudomonas putida KT2440...


  pputida: 11127 features, 8428 with functional annotations
Loading cached genome for Staphylococcus aureus NCTC 8325...


  saureus: 5647 features, 3558 with functional annotations
Loading cached genome for Salmonella enterica Typhimurium LT2...


  senterica: 9497 features, 8475 with functional annotations
Loading cached genome for Mycobacterium tuberculosis H37Rv...


  mtb: 8356 features, 5976 with functional annotations
Loading cached genome for Clostridium acetobutylicum ATCC 824...


  cace: 7776 features, 5692 with functional annotations
Loading cached genome for Synechocystis sp. PCC 6803 substr. GT-I...
  synechocystis: 6788 features, 3566 with functional annotations
Loading cached genome for Klebsiella pneumoniae HS11286...


  kpneumoniae: 11302 features, 8846 with functional annotations
Loading cached genome for Deinococcus radiodurans ATCC 13939...
  dradiodurans: 3177 features, 2183 with functional annotations

Loaded 10/10 genomes


## 4. Build Metabolic Models

In [5]:
atp_medias = load_default_medias()
core_template = templates['Core']
models = {}
build_results = []

for org_id, org_info in organisms.items():
    if org_id not in genomes:
        print(f'Skipping {org_id} (genome not loaded)')
        continue
    
    model_path = DATA_DIR / f'model_{org_id}.json'
    
    if model_path.exists():
        print(f'Loading cached model for {org_info["name"]}...')
        model = cobra.io.load_json_model(str(model_path))
        sol = model.optimize()
        n_genes = len(model.genes)
        print(f'  {org_id}: {len(model.reactions)} reactions, {n_genes} genes, growth={sol.objective_value:.4f}')
        models[org_id] = model
        ms_rxns = [r for r in model.reactions if r.id.startswith('rxn')]
        build_results.append({
            'organism': org_id,
            'name': org_info['name'],
            'template': org_info['template'],
            'phylum': org_info['phylum'],
            'genes': n_genes,
            'total_reactions': len(model.reactions),
            'modelseed_reactions': len(ms_rxns),
            'growth': sol.objective_value,
        })
        continue
    
    print(f'\n=== Building model for {org_info["name"]} ===')
    genome = genomes[org_id]
    template = templates[org_info['template']]
    
    try:
        builder = MSBuilder(genome, template=template, template_core=core_template)
        base_model = cobra.Model(id_or_model=genome.id, name=genome.scientific_name)
        model = builder.build_base_model(base_model, index='0', annotate_with_rast=False)
        MSBuilder.add_atpm(model)
        n_base = len(model.reactions)
        n_genes = len(model.genes)
        print(f'  Base model: {n_base} reactions, {n_genes} genes')
        
        atp_correction = MSATPCorrection(
            model, core_template, atp_medias,
            compartment='c0', atp_hydrolysis_id='ATPM_c0',
            load_default_medias=False,
        )
        atp_correction.evaluate_growth_media()
        atp_correction.determine_growth_media()
        atp_correction.apply_growth_media_gapfilling()
        atp_correction.expand_model_to_genome_scale()
        tests = atp_correction.build_tests()
        n_atp = len(model.reactions) - n_base
        print(f'  ATP correction: +{n_atp} reactions')
        
        complete_media_dict = {}
        for rxn in model.reactions:
            if rxn.id.startswith('EX_'):
                cpd_id = rxn.id.replace('EX_', '').replace('_e0', '')
                complete_media_dict[cpd_id] = 1000
        for cpd in ['cpd00001', 'cpd00067', 'cpd00009', 'cpd00027', 'cpd00007']:
            complete_media_dict[cpd] = 1000
        complete_media = MSMedia.from_dict(complete_media_dict)
        complete_media.id = 'Complete'
        complete_media.name = 'Complete'
        
        gapfill = MSGapfill(
            model, default_gapfill_templates=[template],
            test_conditions=tests, default_target='bio1',
        )
        gapfill_res = gapfill.run_gapfilling(media=complete_media)
        
        gap_sol = {}
        for rxn_id, d in gapfill_res.get('new', {}).items():
            if rxn_id[:-1] in template.reactions:
                gap_sol[rxn_id[:-1]] = get_reaction_constraints_from_direction(d)
        
        model_gf = model.copy()
        MSBuilder.integrate_gapfill_solution(template, model_gf, gap_sol)
        print(f'  Gapfilling: +{len(gap_sol)} reactions')
        
        sol = model_gf.optimize()
        print(f'  Final: {len(model_gf.reactions)} reactions, {len(model_gf.genes)} genes, growth={sol.objective_value:.4f}')
        
        cobra.io.save_json_model(model_gf, str(model_path))
        models[org_id] = model_gf
        ms_rxns = [r for r in model_gf.reactions if r.id.startswith('rxn')]
        build_results.append({
            'organism': org_id,
            'name': org_info['name'],
            'template': org_info['template'],
            'phylum': org_info['phylum'],
            'genes': len(model_gf.genes),
            'total_reactions': len(model_gf.reactions),
            'modelseed_reactions': len(ms_rxns),
            'growth': sol.objective_value,
        })
    except Exception as e:
        print(f'  ERROR building {org_id}: {e}')
        import traceback
        traceback.print_exc()

df_build = pd.DataFrame(build_results)
print(f'\n=== Build Summary ({len(models)} models) ===')
print(df_build.to_string(index=False))

Loading cached model for Escherichia coli K-12 MG1655...


  ecoli: 1765 reactions, 1317 genes, growth=0.5625
Loading cached model for Bacillus subtilis 168...


  bsubtilis: 1345 reactions, 1033 genes, growth=0.5000
Loading cached model for Pseudomonas putida KT2440...


  pputida: 1494 reactions, 1350 genes, growth=0.1667
Loading cached model for Staphylococcus aureus NCTC 8325...


  saureus: 1128 reactions, 777 genes, growth=0.5000
Loading cached model for Salmonella enterica Typhimurium LT2...


...

  mtb: 1150 reactions, 879 genes, growth=0.5000
Loading cached model for Clostridium acetobutylicum ATCC 824...


  cace: 1066 reactions, 819 genes, growth=1.0000
Loading cached model for Synechocystis sp. PCC 6803 substr. GT-I...


  synechocystis: 980 reactions, 719 genes, growth=0.5000
Loading cached model for Klebsiella pneumoniae HS11286...


  kpneumoniae: 1793 reactions, 1601 genes, growth=0.5000
Loading cached model for Deinococcus radiodurans ATCC 13939...


  dradiodurans: 1070 reactions, 712 genes, growth=0.5000

=== Build Summary (10 models) ===
     organism                                    name template         phylum  genes  total_reactions  modelseed_reactions   growth
        ecoli            Escherichia coli K-12 MG1655  GramNeg Proteobacteria   1317             1765                 1574 0.562500
    bsubtilis                   Bacillus subtilis 168  GramPos     Firmicutes   1033             1345                 1224 0.500000
      pputida               Pseudomonas putida KT2440  GramNeg Proteobacteria   1350             1494                 1357 0.166667
      saureus         Staphylococcus aureus NCTC 8325  GramPos     Firmicutes    777             1128                 1018 0.500000
    senterica     Salmonella enterica Typhimurium LT2  GramNeg Proteobacteria   1306             1683                 1518 0.562500
          mtb        Mycobacterium tuberculosis H37Rv  GramPos Actinobacteria    879             1150               

## 5. Load dG Predictions

In [6]:
with open(DATA_DIR / 'dg_predictions_marvin.json') as f:
    marvin_dg = json.load(f)
with open(DATA_DIR / 'dg_predictions_opam2.json') as f:
    opam2_dg = json.load(f)

print(f'Marvin dG: {len(marvin_dg)} reactions')
print(f'OPAM2 dG:  {len(opam2_dg)} reactions')

for org_id, model in models.items():
    ms_rxns = [r.id.replace('_c0', '').replace('_e0', '') for r in model.reactions if r.id.startswith('rxn')]
    n_with_dg = sum(1 for r in ms_rxns if r in marvin_dg)
    print(f'  {org_id:15s}: {len(ms_rxns):>5d} ModelSEED rxns, {n_with_dg:>5d} with dG ({100*n_with_dg/max(len(ms_rxns),1):.0f}%)')

Marvin dG: 31924 reactions
OPAM2 dG:  31924 reactions
  ecoli          :  1574 ModelSEED rxns,  1210 with dG (77%)
  bsubtilis      :  1224 ModelSEED rxns,   941 with dG (77%)
  pputida        :  1357 ModelSEED rxns,  1018 with dG (75%)
  saureus        :  1018 ModelSEED rxns,   775 with dG (76%)
  senterica      :  1518 ModelSEED rxns,  1153 with dG (76%)
  mtb            :  1084 ModelSEED rxns,   797 with dG (74%)
  cace           :   977 ModelSEED rxns,   708 with dG (72%)
  synechocystis  :   917 ModelSEED rxns,   684 with dG (75%)
  kpneumoniae    :  1612 ModelSEED rxns,  1243 with dG (77%)
  dradiodurans   :   996 ModelSEED rxns,   757 with dG (76%)


## 6. Unconstrained FBA + TMFA Comparison

In [7]:
results = []
flux_data = {}  # {org_id: {condition: {rxn_id: flux}}}

for org_id, base_model in models.items():
    print(f'\n=== {org_id}: {organisms[org_id]["name"]} ===')
    flux_data[org_id] = {}
    
    # Unconstrained FBA
    model_unc = base_model.copy()
    sol_unc = model_unc.optimize()
    growth_unc = sol_unc.objective_value if sol_unc.status == 'optimal' else 0.0
    flux_unc = sol_unc.fluxes.to_dict() if sol_unc.status == 'optimal' else {}
    flux_data[org_id]['unconstrained'] = flux_unc
    print(f'  Unconstrained: growth={growth_unc:.4f}')
    
    # TMFA with Marvin dG
    model_marvin = base_model.copy()
    pkg_marvin = TMFAPkg(model_marvin)
    pkg_marvin.build_package({'dg_data': marvin_dg})
    n_constrained_marvin = len(pkg_marvin.variables['dGrxn'])
    sol_marvin = model_marvin.optimize()
    growth_marvin = sol_marvin.objective_value if sol_marvin.status == 'optimal' else 0.0
    flux_marvin = sol_marvin.fluxes.to_dict() if sol_marvin.status == 'optimal' else {}
    flux_data[org_id]['tmfa_marvin'] = flux_marvin
    print(f'  TMFA+Marvin:   growth={growth_marvin:.4f}, {n_constrained_marvin} rxns constrained')
    
    # TMFA with OPAM2 dG
    model_opam2 = base_model.copy()
    pkg_opam2 = TMFAPkg(model_opam2)
    pkg_opam2.build_package({'dg_data': opam2_dg})
    n_constrained_opam2 = len(pkg_opam2.variables['dGrxn'])
    sol_opam2 = model_opam2.optimize()
    growth_opam2 = sol_opam2.objective_value if sol_opam2.status == 'optimal' else 0.0
    flux_opam2 = sol_opam2.fluxes.to_dict() if sol_opam2.status == 'optimal' else {}
    flux_data[org_id]['tmfa_opam2'] = flux_opam2
    print(f'  TMFA+OPAM2:    growth={growth_opam2:.4f}, {n_constrained_opam2} rxns constrained')
    
    results.append({
        'organism': org_id,
        'name': organisms[org_id]['name'],
        'phylum': organisms[org_id]['phylum'],
        'template': organisms[org_id]['template'],
        'total_reactions': len(base_model.reactions),
        'rxns_constrained': n_constrained_marvin,
        'growth_unconstrained': growth_unc,
        'growth_tmfa_marvin': growth_marvin,
        'growth_tmfa_opam2': growth_opam2,
    })

df_results = pd.DataFrame(results)
print('\n=== Growth Comparison ===')
print(df_results[['organism', 'total_reactions', 'rxns_constrained',
                   'growth_unconstrained', 'growth_tmfa_marvin', 'growth_tmfa_opam2']].to_string(index=False))


=== ecoli: Escherichia coli K-12 MG1655 ===


  Unconstrained: growth=0.5625


  TMFA+Marvin:   growth=0.5625, 1210 rxns constrained


  TMFA+OPAM2:    growth=0.5625, 1210 rxns constrained

=== bsubtilis: Bacillus subtilis 168 ===
  Unconstrained: growth=0.5000


  TMFA+Marvin:   growth=0.5000, 941 rxns constrained


...

  Unconstrained: growth=0.5000


  TMFA+Marvin:   growth=0.5000, 1243 rxns constrained


  TMFA+OPAM2:    growth=0.5000, 1243 rxns constrained

=== dradiodurans: Deinococcus radiodurans ATCC 13939 ===
  Unconstrained: growth=0.5000


  TMFA+Marvin:   growth=0.5000, 757 rxns constrained


  TMFA+OPAM2:    growth=0.5000, 757 rxns constrained

=== Growth Comparison ===
     organism  total_reactions  rxns_constrained  growth_unconstrained  growth_tmfa_marvin  growth_tmfa_opam2
        ecoli             1765              1210              0.562500            0.562500           0.562500
    bsubtilis             1345               941              0.500000            0.500000           0.500000
      pputida             1494              1018              0.166667            0.166667           0.166667
      saureus             1128               775              0.500000            0.500000           0.500000
    senterica             1683              1153              0.562500            0.562500           0.562500
          mtb             1150               797              0.500000            0.500000           0.500000
         cace             1066               708              1.000000            1.000000           1.000000
synechocystis              980          

## 7. Per-Organism Flux Diffs: Unconstrained vs TMFA

In [8]:
FLUX_TOL = 1e-6

diff_summary = []
all_flux_diffs = []

for org_id in models:
    flux_unc = flux_data[org_id].get('unconstrained', {})
    flux_marvin = flux_data[org_id].get('tmfa_marvin', {})
    flux_opam2 = flux_data[org_id].get('tmfa_opam2', {})
    
    if not flux_unc or not flux_marvin:
        continue
    
    all_rxns = set(flux_unc.keys())
    ms_rxns = [r for r in all_rxns if r.startswith('rxn')]
    
    # Unconstrained vs TMFA-Marvin
    n_changed_marvin = 0
    n_newly_blocked = 0
    n_newly_active = 0
    n_reversed = 0
    
    for rxn_id in all_rxns:
        v_unc = flux_unc.get(rxn_id, 0)
        v_tmfa = flux_marvin.get(rxn_id, 0)
        diff = abs(v_tmfa - v_unc)
        
        if diff > FLUX_TOL:
            n_changed_marvin += 1
            
            if abs(v_unc) > FLUX_TOL and abs(v_tmfa) <= FLUX_TOL:
                n_newly_blocked += 1
            elif abs(v_unc) <= FLUX_TOL and abs(v_tmfa) > FLUX_TOL:
                n_newly_active += 1
            
            if v_unc * v_tmfa < -FLUX_TOL:
                n_reversed += 1
            
            all_flux_diffs.append({
                'organism': org_id,
                'reaction': rxn_id,
                'flux_unconstrained': v_unc,
                'flux_tmfa_marvin': v_tmfa,
                'flux_diff': v_tmfa - v_unc,
                'abs_diff': diff,
            })
    
    # Marvin vs OPAM2 under TMFA
    n_marvin_opam2_diff = 0
    for rxn_id in all_rxns:
        v_m = flux_marvin.get(rxn_id, 0)
        v_o = flux_opam2.get(rxn_id, 0)
        if abs(v_m - v_o) > FLUX_TOL:
            n_marvin_opam2_diff += 1
    
    diff_summary.append({
        'organism': org_id,
        'name': organisms[org_id]['name'],
        'total_reactions': len(all_rxns),
        'flux_changed_by_tmfa': n_changed_marvin,
        'pct_changed': 100 * n_changed_marvin / max(len(all_rxns), 1),
        'newly_blocked': n_newly_blocked,
        'newly_active': n_newly_active,
        'direction_reversed': n_reversed,
        'marvin_vs_opam2_diffs': n_marvin_opam2_diff,
    })
    
    print(f'{org_id:15s}: {n_changed_marvin:>4d} flux changes ({100*n_changed_marvin/max(len(all_rxns),1):.1f}%), '
          f'{n_newly_blocked} blocked, {n_newly_active} activated, {n_reversed} reversed, '
          f'{n_marvin_opam2_diff} Marvin/OPAM2 diffs')

df_diff_summary = pd.DataFrame(diff_summary)
df_flux_diffs = pd.DataFrame(all_flux_diffs) if all_flux_diffs else pd.DataFrame()

ecoli          :   18 flux changes (1.0%), 4 blocked, 12 activated, 2 reversed, 25 Marvin/OPAM2 diffs
bsubtilis      :    0 flux changes (0.0%), 0 blocked, 0 activated, 0 reversed, 0 Marvin/OPAM2 diffs
pputida        :   24 flux changes (1.6%), 14 blocked, 9 activated, 0 reversed, 17 Marvin/OPAM2 diffs
saureus        :    0 flux changes (0.0%), 0 blocked, 0 activated, 0 reversed, 8 Marvin/OPAM2 diffs
senterica      :   20 flux changes (1.2%), 3 blocked, 13 activated, 4 reversed, 10 Marvin/OPAM2 diffs
mtb            :    0 flux changes (0.0%), 0 blocked, 0 activated, 0 reversed, 0 Marvin/OPAM2 diffs
cace           :   10 flux changes (0.9%), 4 blocked, 6 activated, 0 reversed, 0 Marvin/OPAM2 diffs
synechocystis  :    0 flux changes (0.0%), 0 blocked, 0 activated, 0 reversed, 0 Marvin/OPAM2 diffs
kpneumoniae    :    6 flux changes (0.3%), 0 blocked, 5 activated, 1 reversed, 6 Marvin/OPAM2 diffs
dradiodurans   :    5 flux changes (0.5%), 0 blocked, 4 activated, 1 reversed, 0 Marvin/OPAM2 

## 8. Figures

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Growth rates across organisms x constraint sets
ax = axes[0]
x = np.arange(len(df_results))
width = 0.25
ax.bar(x - width, df_results['growth_unconstrained'], width, label='Unconstrained', color='#2196F3')
ax.bar(x,         df_results['growth_tmfa_marvin'],   width, label='TMFA+Marvin',   color='#FF9800')
ax.bar(x + width, df_results['growth_tmfa_opam2'],    width, label='TMFA+OPAM2',    color='#4CAF50')
ax.set_xticks(x)
ax.set_xticklabels(df_results['organism'], rotation=45, ha='right')
ax.set_ylabel('Growth Rate (1/h)')
ax.set_title('Growth Rates: Unconstrained vs TMFA')
ax.legend()
ax.set_ylim(bottom=0)

# Panel B: TMFA impact summary
ax = axes[1]
ax.barh(df_diff_summary['organism'], df_diff_summary['pct_changed'], color='#9C27B0', alpha=0.7)
ax.set_xlabel('Reactions with Flux Change (%)')
ax.set_title('TMFA Impact on Flux Distribution')
ax.invert_yaxis()

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'diverse_growth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR / "diverse_growth_comparison.png"}')

Saved: /home/freiburgermsu/BERIL-research-observatory/projects/thermodynamic_impact/figures/diverse_growth_comparison.png


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Stacked bar — newly blocked, activated, reversed per organism
ax = axes[0]
x = np.arange(len(df_diff_summary))
ax.bar(x, df_diff_summary['newly_blocked'], label='Newly Blocked', color='#F44336', alpha=0.8)
ax.bar(x, df_diff_summary['newly_active'], bottom=df_diff_summary['newly_blocked'],
       label='Newly Active', color='#4CAF50', alpha=0.8)
ax.bar(x, df_diff_summary['direction_reversed'],
       bottom=df_diff_summary['newly_blocked'] + df_diff_summary['newly_active'],
       label='Direction Reversed', color='#FF9800', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(df_diff_summary['organism'], rotation=45, ha='right')
ax.set_ylabel('Number of Reactions')
ax.set_title('Reaction Classification Changes under TMFA')
ax.legend()

# Panel B: Marvin vs OPAM2 diffs per organism
ax = axes[1]
ax.barh(df_diff_summary['organism'], df_diff_summary['marvin_vs_opam2_diffs'],
        color='#607D8B', alpha=0.8)
ax.set_xlabel('Reactions with Flux Difference')
ax.set_title('Marvin vs OPAM2 Flux Differences under TMFA')
ax.invert_yaxis()

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'diverse_tmfa_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR / "diverse_tmfa_impact.png"}')

Saved: /home/freiburgermsu/BERIL-research-observatory/projects/thermodynamic_impact/figures/diverse_tmfa_impact.png


## 9. Save Data Artifacts

In [11]:
# Growth comparison
df_results.to_csv(DATA_DIR / 'diverse_growth_comparison.tsv', sep='\t', index=False)
print(f'Saved: diverse_growth_comparison.tsv ({len(df_results)} rows)')

# TMFA impact summary
df_diff_summary.to_csv(DATA_DIR / 'diverse_tmfa_summary.tsv', sep='\t', index=False)
print(f'Saved: diverse_tmfa_summary.tsv ({len(df_diff_summary)} rows)')

# Detailed flux diffs
if len(df_flux_diffs) > 0:
    df_flux_diffs.to_csv(DATA_DIR / 'diverse_flux_diffs.tsv', sep='\t', index=False)
    print(f'Saved: diverse_flux_diffs.tsv ({len(df_flux_diffs)} rows)')
else:
    print('No flux differences to save')

# Build summary
df_build.to_csv(DATA_DIR / 'diverse_build_summary.tsv', sep='\t', index=False)
print(f'Saved: diverse_build_summary.tsv ({len(df_build)} rows)')

print('\nDone.')

Saved: diverse_growth_comparison.tsv (10 rows)
Saved: diverse_tmfa_summary.tsv (10 rows)
Saved: diverse_flux_diffs.tsv (83 rows)
Saved: diverse_build_summary.tsv (10 rows)

Done.
